# ChessMamba v3 - Kaggle End-to-End Training

Trains the dual-Mamba + MCTS engine on the **Lichess pre-evaluated positions**
dataset (every position already carries Stockfish analysis: cp / mate / line /
depth). No local Stockfish needed.

| Phase | Accelerator | What it does |
|-------|-------------|--------------|
| 0. Convert data | CPU | Lichess evals -> compact training_data/ numpy arrays |
| 1. Train Eval Mamba | GPU | policy + WDL + contrastive + strategy + action-value |
| 2. Advantage vectors | GPU | phase x context geometric navigation vectors |
| 3. Train Search Mamba | GPU | PV-line look-ahead evaluator |

### Attach these Kaggle datasets
1. **chessmamba-engine** - the engine/ source folder (this repo).
2. **chess-evaluations** (Lichess) - the pre-evaluated positions.

### Training target
Defaults are tuned for a single **8 GB GPU (RTX 3060 Ti)**: d=512 / 16 layers,
AMP + gradient checkpointing, batch 48. They also run on Kaggle's T4. To go
bigger on a larger card: --d-model 768 --n-layer 24 --batch-size 24 --grad-accum 4.

> Tip: run **Phase 0 on a CPU session** (fast, no GPU quota), Save Version to
> persist training_data/ as an output dataset, then attach it to a GPU session
> for Phases 1-3 - or download it and train locally on your 3060 Ti.

## Phase 0 - Data Conversion (CPU session, Internet ON)

In [1]:
!pip install -q python-chess numpy pandas pyarrow zstandard tqdm
print("deps installed")

deps installed



[notice] A new release of pip is available: 26.0.1 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [ ]:
import os, glob, shutil, sys

# Find the engine source and copy it to a writable working dir.
ENGINE_SRC = os.path.dirname(glob.glob('/kaggle/input/**/encoding.py', recursive=True)[0])
ENGINE_DIR = '/kaggle/working/engine'
if os.path.abspath(ENGINE_SRC) != os.path.abspath(ENGINE_DIR):
    shutil.rmtree(ENGINE_DIR, ignore_errors=True)
    shutil.copytree(ENGINE_SRC, ENGINE_DIR)
print('engine ->', ENGINE_DIR)

# Find the Lichess evaluation data, excluding engine files.
EVAL = (glob.glob('/kaggle/input/**/*.parquet', recursive=True)
        + glob.glob('/kaggle/input/**/*.jsonl*', recursive=True)
        + glob.glob('/kaggle/input/**/*.csv', recursive=True))
EVAL = [f for f in EVAL if 'engine' not in f.lower()]
assert EVAL, 'Attach the chess-evaluations dataset!'
EVAL_DIR = os.path.dirname(EVAL[0])
print('eval data ->', EVAL_DIR, '| e.g.', os.path.basename(EVAL[0]))

IndexError: list index out of range

In [ ]:
sys.path.insert(0, ENGINE_DIR)
from encoding import encoder, ACTION_SPACE
print('vocab', encoder.vocab_size(), '| action_space', ACTION_SPACE)

In [ ]:
# Convert. Raise --max-positions for a stronger base (disk/time permitting).
#   ~2-5M positions is a solid Phase-1 target for a single GPU.
import subprocess, sys
OUT = '/kaggle/working/training_data'
cmd = [sys.executable, f'{ENGINE_DIR}/data/convert_lichess_evals.py',
       '--input', EVAL_DIR, '--output', OUT,
       '--max-positions', '3000000', '--min-depth', '20']
print('run:', ' '.join(cmd))
subprocess.run(cmd, check=True, cwd=ENGINE_DIR)

In [ ]:
import numpy as np, json
meta = json.load(open(f'{OUT}/metadata.json')); print(json.dumps(meta, indent=2))
for f in sorted(os.listdir(OUT)):
    print(f'  {f:24s} {os.path.getsize(os.path.join(OUT, f))/1e6:8.1f} MB')
inp = np.load(f'{OUT}/inputs.npy', mmap_mode='r')
print('inputs', inp.shape, '| values are', meta['values_relative_to'])

### (Optional) Save training_data/ for reuse
Click **Save Version** to persist /kaggle/working/training_data as an output
dataset, then attach it to a GPU notebook (or download for local training).

## Phase 1 - Train the Evaluation Mamba (GPU session)

In [ ]:
import torch
print('CUDA', torch.cuda.is_available(),
      torch.cuda.get_device_name(0) if torch.cuda.is_available() else '')

In [ ]:
# Eval Mamba. 3060 Ti-tuned config; adjust --epochs/--batch-size to VRAM.
import subprocess, sys
subprocess.run([sys.executable, f'{ENGINE_DIR}/train.py',
                '--phase', 'eval', '--data', OUT,
                '--epochs', '12', '--batch-size', '48',
                '--d-model', '512', '--n-layer', '16'],
               check=True, cwd=ENGINE_DIR)

## Phase 2 - Geometric Advantage Vectors

In [ ]:
subprocess.run([sys.executable, f'{ENGINE_DIR}/train.py',
                '--phase', 'vectors', '--data', OUT],
               check=True, cwd=ENGINE_DIR)

## Phase 3 - Train the Search Mamba

In [ ]:
subprocess.run([sys.executable, f'{ENGINE_DIR}/train.py',
                '--phase', 'search', '--data', OUT, '--epochs', '8'],
               check=True, cwd=ENGINE_DIR)

## Sanity check - load the model and run MCTS on a position

In [ ]:
import torch, chess, json
from mamba import MambaConfig
from model import ChessMamba
from batched_mcts import BatchedMCTS
from encoding import ACTION_SPACE
dev = 'cuda' if torch.cuda.is_available() else 'cpu'
cfg = MambaConfig(**{k: v for k, v in json.load(open(f'{ENGINE_DIR}/chess_mamba_config.json')).items()
                     if k in MambaConfig.__dataclass_fields__})
model = ChessMamba(cfg, action_space=ACTION_SPACE).to(dev).eval()
model.load_state_dict(torch.load(f'{ENGINE_DIR}/chess_mamba_best.pt', map_location=dev, weights_only=True))
mcts = BatchedMCTS(model, num_simulations=400, batch_size=16, adaptive_sims=True)
b = chess.Board('r1bqkbnr/pppp1ppp/2n5/4p3/4P3/5N2/PPPP1PPP/RNBQKB1R w KQkq - 0 1')
print('best move:', b.san(mcts.search(b)))

## Package artifacts for download / local 3060 Ti

chess_mamba_best.pt (+ _config.json), advantage_vectors.pt, search_mamba.pt
(+ _config.json) are everything the engine needs at inference.

In [ ]:
import shutil, os
art = '/kaggle/working/chessmamba_v3_model'
os.makedirs(art, exist_ok=True)
for f in ['chess_mamba.pt', 'chess_mamba_config.json', 'chess_mamba_best.pt',
          'chess_mamba_best_config.json', 'advantage_vectors.pt',
          'search_mamba.pt', 'search_mamba_config.json']:
    p = os.path.join(ENGINE_DIR, f)
    if os.path.exists(p):
        shutil.copy(p, art)
shutil.make_archive('/kaggle/working/chessmamba_v3_model', 'zip', art)
print('packaged:', os.listdir(art))

## Next steps (local RTX 3060 Ti)

Put chess_mamba_best.pt etc. in engine/ (rename best -> chess_mamba.pt), then:

    # Play via UCI (load into CuteChess / Arena):
    python engine/uci.py

    # Strength check vs Stockfish:
    python engine/tournament.py --opponent stockfish --stockfish <path> \
        --sf-depth 6 --games 40 --sims 400

    # Self-play reinforcement (repeat to pass the supervised ceiling):
    python engine/self_play.py generate --model engine/chess_mamba.pt \
        --games 2000 --sims 400 --out selfplay --device cuda
    python engine/self_play.py train --model engine/chess_mamba.pt \
        --replay selfplay --epochs 2 --device cuda

See engine/NEXT_STEPS.md for the full schedule.